# QMI / QMIPWA S-wave validation

This notebook validates the quasi-model-independent scalar amplitude used by `DalitzPlotFitter`. Magnitude and phase are specified at fixed mass knots and interpolated linearly, following the convention used in the LHCb $D^+\to\pi^-\pi^+\pi^+$ QMIPWA. The published analysis uses 50 non-uniform knots; this validation uses a smaller set so that a complete toy closure fit remains fast.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import DecayChannel, DecayModel, Minimizer, Parameter, QMI, RealImag, Resonance, enable_x64, weighted_resample
enable_x64()

## 1. Truth QMI

The first knot fixes the overall magnitude and phase convention: $a_0=1$ and $\delta_0=0$. All other knot magnitudes and phases are fit parameters.

In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
owner = "pipi_S_qmi"

knots = (0.280, 0.380, 0.520, 0.700, 0.900, 1.120, 1.380, 1.650, 1.720)
truth_mag = np.array([1.00, 1.15, 1.55, 1.20, 0.85, 1.45, 0.95, 0.60, 0.45])
truth_phase = np.array([0.00, 0.20, 0.65, 1.10, 1.65, 2.25, 2.70, 3.05, 3.20])

magnitudes = [1.0]
phases = [0.0]
truth = {}
for i in range(1, len(knots)):
    a = Parameter.dynamics(f"qmi.a{i}", 1.0, owner=owner, bounds=(0.05, 3.0), step=0.02)
    d = Parameter.dynamics(f"qmi.d{i}", 0.0, owner=owner, bounds=(-1.0, 5.5), step=0.03)
    magnitudes.append(a); phases.append(d)
    truth[a.name] = float(truth_mag[i]); truth[d.name] = float(truth_phase[i])

qmi = QMI(knots=knots, magnitudes=tuple(magnitudes), phases=tuple(phases))
model = DecayModel(
    channel,
    [Resonance(owner, pair=(0,1), coefficient=RealImag(1.0,0.0), mass=1.0, width=0.0, spin=0, lineshape=qmi)],
    normalization_resolution=350,
)
print("free QMI parameters =", len(model.parameters))

## 2. Interpolated magnitude, phase and Argand trajectory

In [ ]:
m = jnp.linspace(knots[0], knots[-1], 2000)
qmi_truth = QMI(knots=knots, magnitudes=tuple(truth_mag), phases=tuple(truth_phase))
mag, phase = qmi_truth.interpolated_magnitude_phase(m)
amp = np.asarray(mag * jnp.exp(1j*phase))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
axes[0].plot(np.asarray(m), np.asarray(mag)); axes[0].plot(knots, truth_mag, 'o'); axes[0].set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel="magnitude")
axes[1].plot(np.asarray(m), np.asarray(phase)); axes[1].plot(knots, truth_phase, 'o'); axes[1].set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel="phase [rad]")
axes[2].plot(amp.real, amp.imag); axes[2].plot(truth_mag*np.cos(truth_phase), truth_mag*np.sin(truth_phase), 'o'); axes[2].set(xlabel="Re A", ylabel="Im A"); axes[2].set_aspect('equal', adjustable='datalim')
plt.show()

## 3. Toy MC from the QMI truth model

In [ ]:
N_POOL, N_DATA = 400_000, 50_000
pool = model.generate_phase_space(N_POOL, seed=5100)
norm = model.normalization_sample
pool_cache = model.prepare_cache(pool, norm)
intensity_truth, norm_truth = pool_cache.evaluate(truth)
toy = weighted_resample(jax.random.key(5101), pool, pool.weights*intensity_truth, N_DATA, replace=True)

fig, ax = plt.subplots(figsize=(7,6))
h = ax.hist2d(np.asarray(toy.s12), np.asarray(toy.s13), bins=100)
fig.colorbar(h[3], ax=ax); ax.set(xlabel=r"$s_{12}$ [GeV$^2$]", ylabel=r"$s_{13}$ [GeV$^2$]")
plt.show()

## 4. One randomized closure fit

This is deliberately a single fit. The first knot is fixed to remove the global magnitude/phase ambiguity.

In [ ]:
cache = model.prepare_cache(toy, norm)
def nll(values):
    intensity, normalization = cache.evaluate(values)
    return -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300))) + toy.size*jnp.log(normalization)

rng = np.random.default_rng(5102)
start = {}
for p in model.parameters:
    if '.a' in p.name:
        start[p.name] = float(rng.uniform(0.5, 1.8))
    else:
        start[p.name] = float(rng.uniform(-0.2, 3.8))

minimizer = Minimizer(nll, model.parameters, verbose=1)
print("NLL truth =", float(nll(truth)))
print("NLL start =", float(nll(start)))
result = minimizer.fit(start_values=start, simplex=False, ncall=100000)
fit_values = {p.name: float(result.values[p.name]) for p in model.parameters}
print("valid =", bool(result.valid), "EDM =", float(result.fmin.edm), "DeltaNLL fit-truth =", float(result.fval-nll(truth)))

## 5. Truth, start and fitted knot values

In [ ]:
fit_mag = [1.0] + [fit_values[f"qmi.a{i}"] for i in range(1,len(knots))]
fit_phase = [0.0] + [fit_values[f"qmi.d{i}"] for i in range(1,len(knots))]
start_mag = [1.0] + [start[f"qmi.a{i}"] for i in range(1,len(knots))]
start_phase = [0.0] + [start[f"qmi.d{i}"] for i in range(1,len(knots))]

fig, axes = plt.subplots(1,2,figsize=(13,4.8),constrained_layout=True)
axes[0].plot(knots, truth_mag, 'o-', label='truth'); axes[0].plot(knots, start_mag, 'o--', label='start'); axes[0].plot(knots, fit_mag, 'o-', label='fit'); axes[0].set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel='magnitude'); axes[0].legend()
axes[1].plot(knots, truth_phase, 'o-', label='truth'); axes[1].plot(knots, start_phase, 'o--', label='start'); axes[1].plot(knots, fit_phase, 'o-', label='fit'); axes[1].set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel='phase [rad]'); axes[1].legend()
plt.show()